# REAL-WORLD EVIDENCE INSIGHTS

Goal: The aim of this analysis is to generate real-world evidence insights from NHS prescribing data using the Prescription Cost Analysis (PCA) dataset for January–April 2026. The analysis explores prescribing volumes, prescription costs, and patterns across therapeutic areas (BNF chapters), products, and preparation categories to identify key drivers of NHS prescribing activity and expenditure.

In [ ]:
# setup 
import pandas as pd
import numpy as np

# importing plotting functions library
import sys
sys.path.append("../src")
import Plotting_Functions as pf

# getting the output directory path
from pathlib import Path
Output_dir = Path("../output")

# setting for float display
pd.set_option('display.float_format', '{:,.2f}'.format)


In [ ]:
# loading the data set

# Data preparation note: The RWE analysis uses the cleaned PCA dataset after removal of exception records. These records were excluded as they 
# did not contain standard product-level information required for analysis of medicines, appliances, devices, and therapeutic categories.


df_pca_complete = pd.read_parquet("../data/cleaned data/nhs_prescribing_analysis.parquet")

df_pca_complete.head(3)

In [ ]:
# checking the shape of the dataset

df_pca_complete.shape #2.25 million records, 32 columns

## SECTION 1

### Research Question 1: Which products are the key drivers of NHS prescribing costs?

##### Objective: 

To identify products contributing most significantly to NHS prescribing expenditure.

##### ANALYSIS

Top 10 products by NIC (cost)

In [ ]:
# load the top products dataset

df_nhs_data = pd.read_csv(Output_dir / "Product_Medicine/top_10_products_nic.csv")

df_nhs_data.info()


In [ ]:
# display the data as a horizontal bar chart

pf.plot_barh(df = df_nhs_data, 
          category_col = "generic_bnf_equivalent_name",
          value_col = "nic",
          title = "Top 10 Products by Net Ingredient Cost (NIC)", 
          ylabel = "Product (BNF Chapter)",
          xlabel = "Total NIC (£)",
          isCurrency=True,
          label_currency= "M",
          extra_label = "bnf_chapter"
        ) 


High NHS prescribing expenditure was concentrated among a relatively small number of products, with major contributions from chronic disease management areas. Diabetes-related technologies and metabolic therapies were among the largest cost drivers, alongside respiratory medicines and cardiovascular treatments. Notably, the highest-cost product was a diabetes monitoring appliance (FreeStyle Libre 2 Plus Sensor), demonstrating the contribution of non-medicine products to NHS prescribing expenditure.

In [ ]:
# top 10 products by ITEMS
top_10_items = (
                df_pca_complete.groupby(["bnf_chapter","generic_bnf_equivalent_name"])["items"]
                .sum()
                .sort_values(ascending=False)
                .reset_index()
            )

top_10_items.head(10)

In [ ]:
# visualising the result
# display the data as a horizontal bar chart

pf.plot_barh(df = top_10_items, 
          category_col = "generic_bnf_equivalent_name",
          value_col = "items",
          title = "Top 10 Products by Prescribing Volume", 
          ylabel = "Product (BNF Chapter)",
          xlabel = "Items Dispensed",
          isCurrency=False,
          label_currency= "M",
          extra_label = "bnf_chapter"
        ) 



Routine chronic disease medicines dominate prescribing volume, while a smaller number of advanced therapies and medical technologies disproportionately drive healthcare expenditure.

In [ ]:
# top 20 by ITEM
# output ranked tables and charts

## SECTION 2 - RQ2: Therapeutical Burden (BNF Chapters)

In [ ]:
# % cost by BNF chapter
# % items by BNF chapter
# bar chart
# interpretation of therapeutical areas

In [ ]:
# BNF chapter cost

chapter_cost = (

                    df_pca_complete.groupby("bnf_chapter")["nic"]
                    .sum()
                    .sort_values(ascending=False)
                    .head(10) 
               )

chapter_cost.head(10)


In [ ]:
# visualising the chapter cost data
ax = chapter_cost.sort_values().plot(kind="barh")

ax.xaxis.set_major_formatter(
    ticker.StrMethodFormatter('{x:,.0f}')
)

plt.xlabel("nic")
plt.ylabel("BNF Chapter")
plt.title("Top 10 BNF chapters by cost")

plt.show()

In [ ]:
# BNF chapter items

chapter_items = (

                    df_pca_complete.groupby("bnf_chapter")["items"]
                    .sum()
                    .sort_values(ascending=False)
                    .head(10) 
               )

chapter_items.head(10)


In [ ]:
# visualising the chapter cost data
ax = chapter_items.sort_values().plot(kind="barh")

ax.xaxis.set_major_formatter(
    ticker.StrMethodFormatter('{x:,.0f}')
)

plt.xlabel("items")
plt.ylabel("BNF Chapter")
plt.title("Top 10 BNF chapters by items")

plt.show()

## SECTION 3 - RQ3: Regional/ICB Variation

In [ ]:
# NIC by region
# NIC by ICB
# highlight variation

# ranked regions
# optional heatmap (if you want)

In [ ]:
# Regional cost breakdown by cost

regional_cost_nic = (

    df_pca_complete.groupby("region_name")["nic"]
    .sum()
    .sort_values(ascending=False)

)

regional_cost_nic.head(20)

In [ ]:
# visualising the data

ax = regional_cost_nic.plot(kind="bar")

plt.xlabel("Regions")
plt.ylabel("Cost")
plt.title("Regions by cost")

plt.show()

In [ ]:
# Regional cost breakdown by items

regional_cost_item = (

    df_pca_complete.groupby("region_name")["items"]
    .sum()
    .sort_values(ascending=False)

)

regional_cost_item.head(20)

In [ ]:
# visualising the data

ax = regional_cost_item.plot(kind="bar")

plt.xlabel("Regions")
plt.ylabel("ITems")
plt.title("Regions by Items prescribed")

plt.show()

In [ ]:
# ICB breakdown by cost

icb_nic = (

    df_pca_complete.groupby("icb_code")["nic"]
    .sum()
    .sort_values(ascending=False)

)

icb_nic.head(20)

In [ ]:
# visualising the chapter cost data
ax = icb_nic.sort_values().plot(kind="barh")

ax.xaxis.set_major_formatter(
    ticker.StrMethodFormatter('{x:,.0f}')
)

plt.xlabel("items")
plt.ylabel("BNF Chapter")
plt.title("Top 10 BNF chapters by items")

plt.show()

In [ ]:
# ICB breakdown by items

icb_items = (

    df_pca_complete.groupby("icb_name")["nic"]
    .sum()
    .sort_values(ascending=False)

)

icb_items.head(20)

In [ ]:
# visualising the chapter cost data
ax = icb_items.sort_values().plot(kind="barh")

ax.xaxis.set_major_formatter(
    ticker.StrMethodFormatter('{x:,.0f}')
)

plt.xlabel("items")
plt.ylabel("BNF Chapter")
plt.title("Top 10 BNF chapters by items")

plt.show()

## SECTION 4 - RQ4: Items vs NIC relationship

In [ ]:
# correlation
# scatter plot 
# cost per item


In [ ]:
# is there a strong corelation between items and nic
df_pca_complete[["items","nic"]].corr()  # no strong corelation

In [ ]:
# visualising corelation using a scatter plot

plt.scatter(df_pca_complete["items"], df_pca_complete["nic"])
plt.xlabel("ITEMS")
plt.ylabel("NIC")
plt.title("Prescribing Volume vs Cost")
plt.show()

The analysis identified a weak relationship between prescribing volume and cost, indicating that a relatively small number of low-volume products account for disproportionately high expenditure.

## SECTION 5 - RQ5: Concentration of Cost (Pareto)

In [ ]:
# top drugs by NIC
# CUMULATIVE COST %
# % of cost driven by top 10/20 drugs
# output : pareto chart, strong insight statement

# Is NHS prescribing expenditure concentrated among a small number of products?


In [ ]:
drug_cost = (
    #df_pca_complete.groupby("bnf_presentation_name")["nic"]
    df_pca_complete.groupby("bnf_presentation_code")["nic"]
    .sum()
    .sort_values(ascending=False)
)

drug_cost.head(20)

In [ ]:
#top10_pct = drug_cost.head(10).sum() / drug_cost.sum() * 100
#top10_pct
cumulative_pct = (
    drug_cost.cumsum() / drug_cost.sum()
) * 100


cumulative_pct

In [ ]:
#cumulative_pct.head(10).plot()

In [ ]:
plt.figure(figsize=(12,6))

cumulative_pct.head(10).plot(marker='o')

plt.ylabel("Cumulative % of NIC")
plt.xlabel("Products Ranked by Cost")
plt.title("Cumulative NHS Prescribing Expenditure")

plt.grid(True)

plt.show()

A relatively small number of medicines and healthcare products contributed disproportionately to total NHS prescribing expenditure.

NHS prescribing expenditure was highly concentrated among a relatively small number of medicines and healthcare technologies.

In [ ]:
# prep by cost
prep_nic = (
    df_pca_complete.groupby("prep_class")["nic"]
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
plt.figure(figsize=(8,5))

prep_nic.plot(kind="bar")

plt.ylabel("NIC")
plt.title("NIC by Preparation Class")

plt.tight_layout()

plt.show()

In [ ]:
prep_items = (
    df_pca_complete.groupby("prep_class")["items"]
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
plt.figure(figsize=(8,5))

prep_items.plot(kind="bar")

plt.ylabel("NIC")
plt.title("Items by Preparation Class")

plt.tight_layout()

plt.show()

In [ ]:
df_pca_complete.groupby("year_month")["nic"].sum()

In [ ]:
df_pca_complete.groupby("year_month")["items"].sum()

## SUMMARY

In [ ]:
# 5 key insights
# 1 paragraph summary of prescribing patterns
# limitations
